# A first LaDyS experiment

Fit GPFA to a small synthetic Lorenz dataset, then inspect the learning curves
and reconstructed neural activity.

Open this notebook in Jupyter with LaDyS, NumPy, and Matplotlib installed in the
kernel's Python environment. Run the cells in order. Everything runs on CPU,
and no data download is needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ladys import Experiment, ExperimentConfig, PreprocessingConfig
from ladys.datasets import LorenzDatasetConfig
from ladys.models import GPFAConfig
from ladys.training import TrainerConfig

## Configure the experiment

Lorenz dynamics generate firing rates for 12 simulated neurons; Poisson sampling
turns those rates into spike counts. Training and validation use separate trials
of the same underlying trajectories.

`ExperimentConfig` combines the dataset, model, preprocessing, and training
settings. Here we use raw spike counts and a model with three latent dimensions.
The seed fixes the generated data and the training randomness.

In [ ]:
config = ExperimentConfig(
    dataset=LorenzDatasetConfig(
        neurons=12,
        num_inits=2,
        num_trials=8,
        num_steps=60,
        seed=0,
    ),
    model=GPFAConfig(latent_dim=3),
    preprocessing=PreprocessingConfig(),
    trainer=TrainerConfig(epochs=20, device="cpu"),
    batch_size=4,
    output_dir="runs/lorenz_tutorial",
)

## Train and evaluate

`Experiment.run()` prepares the data, builds and trains the model, evaluates it
on validation trials, and saves the run. The returned result contains the
learning history, evaluation metrics, and artifact paths.

In [ ]:
result = Experiment(config).run()

## Plot learning curves

GPFA minimizes negative log marginal likelihood. Lower loss is better; the
validation curve shows how the model fits trials it did not train on.

In [ ]:
epochs = [report.epoch + 1 for report in result.history]
train_loss = [report.train.loss for report in result.history]
valid_loss = [report.valid.loss for report in result.history]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(epochs, train_loss, label="Training")
ax.plot(epochs, valid_loss, label="Validation")
ax.set(xlabel="Epoch", ylabel="Negative log marginal likelihood", title="GPFA learning curves")
ax.legend()
fig.tight_layout()
plt.show()

## Inspect reconstructed activity

The saved predictions contain the model's estimated rates and the known Lorenz
rates for the validation trials. Compare them for one neuron in the first
validation trial. Ground-truth rates are used for evaluation, not training.

In [ ]:
with np.load(result.predictions_path) as predictions:
    true_rates = predictions["target_rates"][0, :, 0]
    predicted_rates = predictions["pred_rates"][0, :, 0]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(true_rates, label="True rate")
ax.plot(predicted_rates, label="GPFA estimate")
ax.set(xlabel="Time bin", ylabel="Firing rate (Hz)", title="Validation trial 1, neuron 1")
ax.legend()
fig.tight_layout()
plt.show()

## Inspect the results

The run folder includes `config.json`, `history.csv`, `metrics.json`,
`model.pt`, and `predictions.npz`. The metrics below include rate reconstruction
error and latent reconstruction quality.

In [ ]:
print(f"Run saved to: {result.run_dir.resolve()}")
result.metrics

Try changing `latent_dim` or `epochs` and rerun from the configuration cell.
For other models and configuration options, see the
[LaDyS documentation](https://zkunkworks.com/ladys/).